# DCT Probe Analysis v2

Clean re-analysis with calibrated directional probes and ensemble methods.

**Experiments:**
1. Individual probe comparison — AUROC & TPR@1%FPR bar charts + ROC curves (MM, LR, Top-10 DCT)
2. Generalization across datasets — MM, LR, Best DCT direction
3. Ensemble k-sweep — find optimal k for mean/max ensemble probes
4. Ensemble generalization across datasets

In [30]:
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from IPython.display import display

In [31]:
import os
# Ensure working directory is repo root so relative paths like "dct_probes/..." resolve correctly
repo_root = Path(__file__).parent.parent if "__file__" in dir() else Path.cwd()
while not (repo_root / "dct_probes").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
os.chdir(repo_root)
print(f"Working directory: {Path.cwd()}")

Working directory: /Users/FletcAW1/Documents/repos/personal/SPAR-causal-probes


In [32]:
EXPERIMENT_NAME = "llama-3.1-8b-32samples-fewshot-lying-categorical"
experiment_dir = Path(f"dct_probes/experiments/{EXPERIMENT_NAME}")
data_dir = Path("dct_probes/data")

with open(experiment_dir / "vectors" / "dct_run_config.json") as f:
    run_cfg = json.load(f)

MODEL_NAME = run_cfg["MODEL_NAME"]
PROBE_LAYER = run_cfg["PROBE_LAYER"]
SYSTEM_PROMPT = run_cfg.get("SYSTEM_PROMPT", "You are a helpful assistant")
TOKEN_IDXS_START = run_cfg.get("TOKEN_IDXS_START", -3)
TOKEN_IDXS_STOP = run_cfg.get("TOKEN_IDXS_STOP", None)
INPUT_SCALE = run_cfg.get("INPUT_SCALE", None)

device = torch.device(
    "mps" if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"Device:      {device}")
print(f"Experiment:  {EXPERIMENT_NAME}")
print(f"Model:       {MODEL_NAME}")
print(f"Probe layer: {PROBE_LAYER}")

Device:      mps
Experiment:  llama-3.1-8b-32samples-fewshot-lying-categorical
Model:       meta-llama/Llama-3.1-8B-Instruct
Probe layer: 10


In [33]:
PLOT_COLORS = {
    "mm": "#7C3AED",
    "lr": "#D97706",
    "dct": "#0891B2",
    "ensemble_mean": "#16A34A",
    "ensemble_max": "#DC2626",
    "train": "#2563EB",
    "test": "#16A34A",
    "random": "#6B7280",
}

PLOT_BASE = dict(
    template="plotly_white",
    font=dict(family="Arial, sans-serif", size=14),
    title_font=dict(family="Arial, sans-serif", size=18),
    margin=dict(l=70, r=40, t=80, b=60),
    legend=dict(bgcolor="rgba(255,255,255,0.8)", bordercolor="#E5E7EB", borderwidth=1),
)

# DCT direction color gradient (light to dark cyan/blue)
DCT_COLORS = [
    "#0e7490", "#0891b2", "#06b6d4", "#22d3ee", "#67e8f9",
    "#1d4ed8", "#2563eb", "#3b82f6", "#60a5fa", "#93c5fd",
]

In [34]:
IMAGE_DIR = Path("dct_probes/images") / EXPERIMENT_NAME
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name: str, scale: int = 2) -> None:
    path = IMAGE_DIR / f"{name}.png"
    fig.write_image(str(path), scale=scale)
    print(f"Saved: {path}")

print(f"Image output dir: {IMAGE_DIR}")

Image output dir: dct_probes/images/llama-3.1-8b-32samples-fewshot-lying-categorical


## Load DCT Vectors & Judge Results

In [35]:
data = torch.load(experiment_dir / "vectors" / "dct_vectors.pt", weights_only=True, map_location="cpu")
V = data["V"].to(dtype=torch.float32)
print(f"V shape: {V.shape}  (d_model x num_factors)")

df_judge = pd.read_json(experiment_dir / "results" / "judge_results.jsonl", lines=True)

if "judge_category" in df_judge.columns:
    # Categorical schema: score = P(CONFIDENTLY_WRONG) / P(CONFIDENTLY_WRONG | GARBAGE | REFUSAL)
    def lying_score(cats):
        cw  = (cats == "CONFIDENTLY_WRONG").mean()
        ca  = (cats == "CORRECT").mean()
        hw  = (cats == "HEDGED_WRONG").mean()
        gc_ = (cats == "GARBAGE").mean()
        ref = (cats == "REFUSAL").mean()
        return cw / (cw + gc_ + ref + ca + hw + 1e-8)

    steered = df_judge[df_judge["factor_idx"] >= 0].copy()
    mean_deltas = steered.groupby("factor_idx")["judge_category"].apply(lying_score).sort_values(ascending=False)

    baseline_cats = df_judge[df_judge["factor_idx"] == -1]["judge_category"].value_counts(normalize=True)
    print("Baseline category distribution:")
    print(baseline_cats.to_string())
else:
    # Numeric schema: rank by deception-score delta vs baseline
    df_judge = df_judge.dropna(subset=["judge_score"])
    df_judge["judge_score"] = df_judge["judge_score"].astype(int)

    baseline = df_judge[df_judge["factor_idx"] == -1].set_index("prompt_id")["judge_score"]
    steered = df_judge[df_judge["factor_idx"] >= 0].copy()
    steered = steered.merge(baseline.rename("baseline_score"), on="prompt_id")
    steered["delta"] = steered["judge_score"] - steered["baseline_score"]
    mean_deltas = steered.groupby("factor_idx")["delta"].mean().sort_values(ascending=False)

    print(f"Mean baseline deception score: {baseline.mean():.1f}")

print(f"\nNumber of DCT factors: {len(mean_deltas)}")
print("\nTop 10 factor indices:")
print(mean_deltas.head(10).to_string())

V shape: torch.Size([4096, 512])  (d_model x num_factors)
Baseline category distribution:
judge_category
GARBAGE    0.5
CORRECT    0.5

Number of DCT factors: 512

Top 10 factor indices:
factor_idx
435    0.4
62     0.3
72     0.3
218    0.3
24     0.3
375    0.3
431    0.3
198    0.3
249    0.3
484    0.3


In [36]:
if "judge_category" in df_judge.columns:
    # ── Colour palette ─────────────────────────────────────────────────────────
    CAT_COLORS = {
        "CONFIDENTLY_WRONG": "#EF4444",  # red  → right side
        "CORRECT":           "#22C55E",  # green
        "GARBAGE":           "#F59E0B",  # amber
        "REFUSAL":           "#8B5CF6",  # purple
    }
    LEFT_CATS = ["CORRECT", "GARBAGE", "REFUSAL"]  # stacked on left
    ALL_CATS  = list(CAT_COLORS.keys())

    def cat_dist(df_sub):
        vc = df_sub["judge_category"].value_counts(normalize=True)
        return {c: float(vc.get(c, 0.0)) for c in ALL_CATS}

    # Baseline (unsteered, factor_idx == -1) + top-10 factors
    baseline_dist = cat_dist(df_judge[df_judge["factor_idx"] == -1])
    top10_idx     = mean_deltas.head(10).index.tolist()
    factor_dists  = [cat_dist(steered[steered["factor_idx"] == idx]) for idx in top10_idx]

    # Row labels — baseline at top, top-10 below
    # (plotly draws horizontal bars bottom-to-top, so reverse so DCT #10 is at the bottom)
    row_labels = (
        [f"DCT #{i+1}  (idx {idx})" for i, idx in enumerate(top10_idx)][::-1]
        + ["── Baseline (unsteered) ──"]
    )
    all_dists = factor_dists[::-1] + [baseline_dist]

    tick_vals = [-1.0, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1.0]
    tick_text = ["100%", "75%",  "50%", "25%", "0%", "25%", "50%", "75%", "100%"]

    fig = go.Figure()

    # Right side: CONFIDENTLY_WRONG
    fig.add_trace(go.Bar(
        y=row_labels,
        x=[d["CONFIDENTLY_WRONG"] for d in all_dists],
        orientation="h",
        name="CONFIDENTLY_WRONG",
        marker_color=CAT_COLORS["CONFIDENTLY_WRONG"],
        hovertemplate="%{y}<br>CONFIDENTLY_WRONG: %{x:.1%}<extra></extra>",
    ))

    # Left side: other categories stacked (negative x, real value in customdata)
    for cat in LEFT_CATS:
        fig.add_trace(go.Bar(
            y=row_labels,
            x=[-d[cat] for d in all_dists],
            orientation="h",
            name=cat,
            marker_color=CAT_COLORS[cat],
            customdata=[d[cat] for d in all_dists],
            hovertemplate="%{y}<br>" + cat + ": %{customdata:.1%}<extra></extra>",
        ))

    # Separator line between baseline row (index 10) and DCT #1 row (index 9)
    fig.add_shape(
        type="line",
        xref="paper", x0=0, x1=1,
        yref="y", y0=9.5, y1=9.5,
        line=dict(color="#9CA3AF", width=1.5, dash="dot"),
    )

    layout = {**PLOT_BASE}
    layout["legend"] = dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    fig.update_layout(
        **layout,
        barmode="relative",
        title=(
            "Judge Category Distribution — Baseline vs Top-10 DCT factors<br>"
        ),
        xaxis=dict(
            title="Proportion of responses",
            tickvals=tick_vals,
            ticktext=tick_text,
            range=[-1.05, 1.05],
            zeroline=True,
            zerolinecolor="#374151",
            zerolinewidth=2,
        ),
        height=520,
        width=960,
    )
    fig.show()
    save_fig(fig, "judge_category_butterfly")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Saved: dct_probes/images/llama-3.1-8b-32samples-fewshot-lying-categorical/judge_category_butterfly.png


## Extract Activations

Load model and extract activations for GoT datasets, or load from cache.

In [37]:
DATASET_NAMES = ["cities", "sp_en_trans", "larger_than"]
DATASET_DISPLAY = {"cities": "Cities", "sp_en_trans": "Spanish", "larger_than": "Larger Than"}

datasets = {}
for name in DATASET_NAMES:
    df = pd.read_csv(data_dir / "got_datasets" / f"{name}.csv")
    datasets[name] = df
    n_pos = df["label"].sum()
    print(f"{name}: {len(df)} samples ({int(n_pos)} positive, {len(df)-int(n_pos)} negative)")

cities: 1496 samples (748 positive, 748 negative)
sp_en_trans: 354 samples (177 positive, 177 negative)
larger_than: 1980 samples (990 positive, 990 negative)


In [38]:
def extract_activations(
    statements: list[str],
    model,
    tokenizer,
    layer: int,
    system_prompt: str = "You are a helpful assistant",
    token_idxs_start: int = -3,
    token_idxs_stop: int | None = None,
    batch_size: int = 4,
) -> torch.Tensor:
    """Extract mean activations at a given layer for each statement."""
    all_acts = []
    for stmt in tqdm(statements, desc=f"Layer {layer}"):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": stmt},
        ]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        with torch.no_grad():
            out = model(inputs, output_hidden_states=True)
        hidden = out.hidden_states[layer]  # (1, seq_len, d_model)
        act = hidden[0, token_idxs_start:token_idxs_stop, :].mean(dim=0).float().cpu()
        all_acts.append(act)
    return torch.stack(all_acts)


cache_path = experiment_dir / "vectors" / "activations_cache.pt"

if cache_path.exists():
    print("Loading cached activations...")
    cache = torch.load(cache_path, weights_only=True, map_location="cpu")
    activations = cache["activations"]
    labels_dict = cache["labels"]
    print("Done.")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    print(f"Loading model {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
    )
    model.eval()
    print("Model loaded. Extracting activations...")

    activations = {}
    labels_dict = {}
    for name in DATASET_NAMES:
        df = datasets[name]
        statements = df["statement"].tolist()
        labels = torch.tensor(df["label"].values, dtype=torch.float32)
        acts = extract_activations(
            statements, model, tokenizer, PROBE_LAYER,
            system_prompt=SYSTEM_PROMPT,
            token_idxs_start=TOKEN_IDXS_START,
            token_idxs_stop=TOKEN_IDXS_STOP,
        )
        activations[name] = acts
        labels_dict[name] = labels
        print(f"  {name}: {acts.shape}")

    torch.save({"activations": activations, "labels": labels_dict}, cache_path)
    print(f"\nSaved cache to {cache_path}")

    del model
    gc.collect()

for name in DATASET_NAMES:
    print(f"{name}: acts {activations[name].shape}, labels {labels_dict[name].shape}")

Loading model meta-llama/Llama-3.1-8B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded. Extracting activations...


Layer 10: 100%|██████████| 1496/1496 [04:47<00:00,  5.21it/s]


  cities: torch.Size([1496, 4096])


Layer 10: 100%|██████████| 354/354 [01:07<00:00,  5.25it/s]


  sp_en_trans: torch.Size([354, 4096])


Layer 10: 100%|██████████| 1980/1980 [06:43<00:00,  4.91it/s]


  larger_than: torch.Size([1980, 4096])

Saved cache to dct_probes/experiments/llama-3.1-8b-32samples-fewshot-lying-categorical/vectors/activations_cache.pt
cities: acts torch.Size([1496, 4096]), labels torch.Size([1496])
sp_en_trans: acts torch.Size([354, 4096]), labels torch.Size([354])
larger_than: acts torch.Size([1980, 4096]), labels torch.Size([1980])


In [39]:
torch.manual_seed(42)
train_acts: dict[str, torch.Tensor] = {}
test_acts: dict[str, torch.Tensor] = {}
train_labels: dict[str, torch.Tensor] = {}
test_labels: dict[str, torch.Tensor] = {}

for name in DATASET_NAMES:
    acts = activations[name]
    labs = labels_dict[name]
    n = len(acts)
    perm = torch.randperm(n)
    n_train = int(0.8 * n)
    train_acts[name] = acts[perm[:n_train]]
    test_acts[name] = acts[perm[n_train:]]
    train_labels[name] = labs[perm[:n_train]]
    test_labels[name] = labs[perm[n_train:]]
    print(f"{name}: {n_train} train / {n - n_train} test")

cities: 1196 train / 300 test
sp_en_trans: 283 train / 71 test
larger_than: 1584 train / 396 test


## Probe Implementations

In [40]:
class Probe(nn.Module):
    pass


class DirectionalProbe(Probe):
    """Directional probe for transformer activations.

    Direction is fixed; magnitude and bias are trainable for calibration.
    """

    def __init__(self, direction: torch.Tensor):
        super().__init__()
        if direction.dim() == 1:
            direction = direction.unsqueeze(-1)
        direction = direction / torch.norm(direction, dim=0, keepdim=True)
        self.direction = nn.Parameter(direction, requires_grad=False)
        self.magnitude = nn.Parameter(torch.tensor(1.0), requires_grad=True)
        self.bias = nn.Parameter(torch.tensor(0.0), requires_grad=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.matmul(x, self.direction * self.magnitude).squeeze(-1) + self.bias


class FixedEnsembleProbe(Probe):
    """Fixed ensemble of directional probes using max or mean aggregation.

    Directions are fixed; per-direction magnitudes and biases are trainable.
    """

    def __init__(self, directions: torch.Tensor, aggregation: str = "max"):
        super().__init__()
        if directions.dim() == 1:
            directions = directions.unsqueeze(1)
        directions = directions / torch.norm(directions, dim=0, keepdim=True)
        self.directions = nn.Parameter(directions, requires_grad=False)
        self.k = directions.shape[1]
        self.magnitudes = nn.Parameter(torch.ones(self.k), requires_grad=True)
        self.biases = nn.Parameter(torch.zeros(self.k), requires_grad=True)
        if aggregation not in ["max", "mean"]:
            raise ValueError("Aggregation must be 'max' or 'mean'")
        self.aggregation = aggregation

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        projections = torch.matmul(x, self.directions)  # (n, k)
        logits = projections * self.magnitudes + self.biases  # (n, k)
        if self.aggregation == "max":
            return logits.max(dim=-1).values
        else:
            return logits.mean(dim=-1)


class MMProbe(Probe):
    """Difference-of-means probe."""

    def __init__(self, direction: torch.Tensor, covariance: torch.Tensor | None = None, atol: float = 1e-3):
        super().__init__()
        self.direction = nn.Parameter(direction, requires_grad=False)
        if covariance is not None:
            inv = torch.linalg.pinv(covariance, hermitian=True, atol=atol)
            self.inv = nn.Parameter(inv, requires_grad=False)
        else:
            self.inv = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x.float() @ self.direction

    @staticmethod
    def from_data(acts: torch.Tensor, labels: torch.Tensor, device: str = "cpu") -> "MMProbe":
        acts = acts.float()
        pos_acts = acts[labels == 1]
        neg_acts = acts[labels == 0]
        pos_mean = pos_acts.mean(0)
        neg_mean = neg_acts.mean(0)
        direction = pos_mean - neg_mean
        centered = torch.cat([pos_acts - pos_mean, neg_acts - neg_mean], dim=0)
        covariance = centered.T @ centered / acts.shape[0]
        return MMProbe(direction, covariance=covariance).to(device)


class LRProbe(Probe):
    """Logistic regression probe with StandardScaler normalisation."""

    def __init__(self, d_in: int, scaler_mean=None, scaler_scale=None):
        super().__init__()
        self.linear = nn.Linear(d_in, 1, bias=False)
        self.register_buffer("scaler_mean", scaler_mean)
        self.register_buffer("scaler_scale", scaler_scale)

    def _normalize(self, x: torch.Tensor) -> torch.Tensor:
        if self.scaler_mean is not None:
            return (x - self.scaler_mean) / self.scaler_scale
        return x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(self._normalize(x.float())).squeeze(-1)

    @staticmethod
    def from_data(acts: torch.Tensor, labels: torch.Tensor, C: float = 0.1, device: str = "cpu") -> "LRProbe":
        X = acts.cpu().float().numpy()
        y = labels.cpu().float().numpy()
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        lr_model = LogisticRegression(C=C, random_state=42, fit_intercept=False, max_iter=1000)
        lr_model.fit(X_scaled, y)
        scaler_mean = torch.tensor(scaler.mean_, dtype=torch.float32)
        scaler_scale = torch.tensor(scaler.scale_, dtype=torch.float32)
        probe = LRProbe(acts.shape[-1], scaler_mean=scaler_mean, scaler_scale=scaler_scale).to(device)
        probe.linear.weight.data[0] = torch.tensor(lr_model.coef_[0], dtype=torch.float32)
        return probe

In [41]:
def train_probe(
    probe: Probe,
    acts: torch.Tensor,
    labels: torch.Tensor,
    lr: float = 0.01,
    n_epochs: int = 500,
) -> None:
    """Train all requires_grad parameters (magnitude/bias) via BCE loss."""
    trainable = [p for p in probe.parameters() if p.requires_grad]
    if not trainable:
        return
    acts = acts.float()
    labels = labels.float()
    optimizer = torch.optim.Adam(trainable, lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    probe.train()
    for _ in range(n_epochs):
        optimizer.zero_grad()
        loss = criterion(probe(acts), labels)
        loss.backward()
        optimizer.step()
    probe.eval()


def tpr_at_fpr(y_true: np.ndarray, y_scores: np.ndarray, target_fpr: float = 0.01) -> float:
    """Interpolated TPR at a given FPR."""
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    idx = np.searchsorted(fpr, target_fpr, side="right")
    return float(tpr[min(idx, len(tpr) - 1)])


def eval_probe(probe: Probe, acts: torch.Tensor, labels: torch.Tensor) -> dict:
    """Return AUROC, TPR@1%FPR, raw scores, and full ROC curve."""
    probe.eval()
    with torch.no_grad():
        scores = probe(acts.float()).cpu().numpy()
    y = labels.cpu().numpy()
    fpr_vals, tpr_vals, _ = roc_curve(y, scores)
    return {
        "auroc": roc_auc_score(y, scores),
        "tpr@1fpr": tpr_at_fpr(y, scores, target_fpr=0.01),
        "scores": scores,
        "fpr": fpr_vals,
        "tpr": tpr_vals,
    }

---
## Experiment 1: Individual Probe Comparison

Train MM, LR, and the Top-10 DCT directional probes on the **Cities** training set.
Evaluate AUROC and TPR@1%FPR on the Cities test set.
Note: DCT directions are fixed; only magnitude and bias are calibrated on training data.

In [42]:
TOP_K_DISPLAY = 10

# --- Baseline probes (Cities train) ---
mm_probe = MMProbe.from_data(train_acts["cities"], train_labels["cities"])
lr_probe = LRProbe.from_data(train_acts["cities"], train_labels["cities"])

# --- Rank all DCT directions by judge score, pick top-K ---
all_indices_by_judge = mean_deltas.index.tolist()
top_indices_display = all_indices_by_judge[:TOP_K_DISPLAY]

# Compute train AUROCs for the selected directions (for display only)
dct_train_aurocs: dict[int, float] = {}
for idx in top_indices_display:
    direction = V[:, idx].float()
    probe = DirectionalProbe(direction)
    train_probe(probe, train_acts["cities"], train_labels["cities"])
    dct_train_aurocs[idx] = eval_probe(probe, train_acts["cities"], train_labels["cities"])["auroc"]

dct_probes: dict[str, DirectionalProbe] = {}
for rank, idx in enumerate(top_indices_display):
    direction = V[:, idx].float()
    probe = DirectionalProbe(direction)
    train_probe(probe, train_acts["cities"], train_labels["cities"])
    dct_probes[f"DCT #{rank + 1}"] = probe

print(f"Trained MM, LR, and {TOP_K_DISPLAY} DCT directional probes.")
print(f"DCT indices (ranked by judge score): {top_indices_display}")
print(f"Train AUROCs: {[round(dct_train_aurocs[i], 3) for i in top_indices_display]}")
print(f"Judge scores: {[round(float(mean_deltas[i]), 3) for i in top_indices_display]}")

Trained MM, LR, and 10 DCT directional probes.
DCT indices (ranked by judge score): [435, 62, 72, 218, 24, 375, 431, 198, 249, 484]
Train AUROCs: [0.535, 0.646, 0.904, 0.435, 0.446, 0.908, 0.948, 0.965, 0.646, 0.662]
Judge scores: [0.4, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3]


In [43]:
# Evaluate all probes on Cities test set
results: dict[str, dict] = {}
results["MM"] = eval_probe(mm_probe, test_acts["cities"], test_labels["cities"])
results["LR"] = eval_probe(lr_probe, test_acts["cities"], test_labels["cities"])
for name, probe in dct_probes.items():
    results[name] = eval_probe(probe, test_acts["cities"], test_labels["cities"])

# Summary table
rows = [
    {"Probe": name, "AUROC": f"{r['auroc']:.3f}", "TPR@1%FPR": f"{r['tpr@1fpr']:.3f}"}
    for name, r in results.items()
]
display(pd.DataFrame(rows))

,Probe,AUROC,TPR@1%FPR
0,MM,0.996,0.898
1,LR,1.000,0.986
2,DCT #1,0.557,0.020
3,DCT #2,0.611,0.027
4,DCT #3,0.905,0.259
5,DCT #4,0.428,0.014
6,DCT #5,0.429,0.007
7,DCT #6,0.896,0.102
8,DCT #7,0.933,0.469
9,DCT #8,0.960,0.061


In [44]:
probe_names = list(results.keys())
aurocs = [results[n]["auroc"] for n in probe_names]
tprs = [results[n]["tpr@1fpr"] for n in probe_names]

bar_colors = []
for n in probe_names:
    if n == "MM":
        bar_colors.append(PLOT_COLORS["mm"])
    elif n == "LR":
        bar_colors.append(PLOT_COLORS["lr"])
    else:
        bar_colors.append(PLOT_COLORS["dct"])

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["AUROC", "TPR @ 1% FPR"],
    horizontal_spacing=0.12,
)

fig.add_trace(go.Bar(
    x=probe_names, y=aurocs,
    marker_color=bar_colors, showlegend=False,
    text=[f"{v:.3f}" for v in aurocs], textposition="outside",
), row=1, col=1)

fig.add_trace(go.Bar(
    x=probe_names, y=tprs,
    marker_color=bar_colors, showlegend=False,
    text=[f"{v:.3f}" for v in tprs], textposition="outside",
), row=1, col=2)

fig.update_layout(
    **PLOT_BASE,
    title="Experiment 1: Individual Probe Performance (Test Set — Cities)",
    height=480, width=1100,
    yaxis=dict(range=[0, 1.15], title="AUROC"),
    yaxis2=dict(range=[0, 1.15], title="TPR @ 1% FPR"),
)
fig.show()
save_fig(fig, "exp1_probe_performance")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Saved: dct_probes/images/llama-3.1-8b-32samples-fewshot-lying-categorical/exp1_probe_performance.png


In [45]:
fig = go.Figure()

# MM
r = results["MM"]
fig.add_trace(go.Scatter(
    x=r["fpr"], y=r["tpr"], mode="lines",
    name=f"MM (AUROC={r['auroc']:.3f})",
    line=dict(color=PLOT_COLORS["mm"], width=2.5),
))

# LR
r = results["LR"]
fig.add_trace(go.Scatter(
    x=r["fpr"], y=r["tpr"], mode="lines",
    name=f"LR (AUROC={r['auroc']:.3f})",
    line=dict(color=PLOT_COLORS["lr"], width=2.5),
))

# Top-10 DCT probes
for i, name in enumerate(dct_probes):
    r = results[name]
    fig.add_trace(go.Scatter(
        x=r["fpr"], y=r["tpr"], mode="lines",
        name=f"{name} ({r['auroc']:.3f})",
        line=dict(color=DCT_COLORS[i % len(DCT_COLORS)], width=1.5, dash="dot"),
        opacity=0.8,
    ))

# Diagonal
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines",
    line=dict(color="gray", width=1, dash="dash"), showlegend=False,
))

# FPR=1% reference line
fig.add_vline(x=0.01, line_dash="dot", line_color="black", opacity=0.4,
              annotation_text="1% FPR", annotation_position="top right",
              annotation_font_size=11)

fig.update_layout(
    **PLOT_BASE,
    title="Experiment 1: ROC Curves (Test Set — Cities)",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    height=520, width=780,
)
fig.show()
save_fig(fig, "exp1_roc_curves")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Saved: dct_probes/images/llama-3.1-8b-32samples-fewshot-lying-categorical/exp1_roc_curves.png


---
## Experiment 2: Generalization Across Datasets

For each training dataset (Cities, Spanish, Larger-Than), train MM, LR, and the best DCT direction probe, then evaluate on all three test datasets.

The best DCT direction is the top-ranked factor by mean lying delta. Its direction is fixed; magnitude/bias are recalibrated on each training set.

In [46]:
best_dct_idx = top_indices_display[0]
best_direction = V[:, best_dct_idx].float()
print(f"Best DCT factor (by judge score): #{best_dct_idx} (judge score = {float(mean_deltas[best_dct_idx]):.3f})")

TOP_K_GEN = 10  # search within top-k judge-ranked directions for generalization

def best_dct_probe_for(acts: torch.Tensor, labels: torch.Tensor) -> DirectionalProbe:
    """Pick the best-AUROC direction from the top-K judge-ranked factors, calibrated on this dataset."""
    candidates = all_indices_by_judge[:TOP_K_GEN]
    best_idx, best_auroc = -1, -1.0
    for idx in candidates:
        probe = DirectionalProbe(V[:, idx].float())
        train_probe(probe, acts, labels)
        auroc = eval_probe(probe, acts, labels)["auroc"]
        if auroc > best_auroc:
            best_auroc, best_idx = auroc, idx
    probe = DirectionalProbe(V[:, best_idx].float())
    train_probe(probe, acts, labels)
    print(f"  Best DCT factor (top-{TOP_K_GEN} judge): #{best_idx} (train AUROC = {best_auroc:.3f})")
    return probe


def build_generalization_data(
    probe_factory,
    train_acts: dict,
    train_labels: dict,
    test_acts: dict,
    test_labels: dict,
    dataset_names: list[str],
) -> dict:
    """Train a probe per dataset and evaluate on all datasets."""
    gen = {}
    for train_name in dataset_names:
        probe = probe_factory(train_acts[train_name], train_labels[train_name])
        gen[train_name] = {
            test_name: eval_probe(probe, test_acts[test_name], test_labels[test_name])
            for test_name in dataset_names
        }
    return gen


gen_mm = build_generalization_data(
    lambda a, l: MMProbe.from_data(a, l),
    train_acts, train_labels, test_acts, test_labels, DATASET_NAMES,
)

gen_lr = build_generalization_data(
    lambda a, l: LRProbe.from_data(a, l),
    train_acts, train_labels, test_acts, test_labels, DATASET_NAMES,
)

print(f"DCT best directions per dataset (search over top-{TOP_K_GEN} by judge):")
gen_dct = build_generalization_data(
    best_dct_probe_for,
    train_acts, train_labels, test_acts, test_labels, DATASET_NAMES,
)

print("Generalization data computed.")

Best DCT factor (by judge score): #435 (judge score = 0.400)
DCT best directions per dataset (search over top-10 by judge):
  Best DCT factor (top-10 judge): #198 (train AUROC = 0.965)
  Best DCT factor (top-10 judge): #431 (train AUROC = 0.795)
  Best DCT factor (top-10 judge): #431 (train AUROC = 0.809)
Generalization data computed.


In [47]:
def gen_to_matrix(gen_data: dict, metric: str, dataset_names: list[str]) -> np.ndarray:
    mat = np.zeros((len(dataset_names), len(dataset_names)))
    for i, train_name in enumerate(dataset_names):
        for j, test_name in enumerate(dataset_names):
            mat[i, j] = gen_data[train_name][test_name][metric]
    return mat


labels_display = [DATASET_DISPLAY[n] for n in DATASET_NAMES]

for metric, metric_title in [("auroc", "AUROC"), ("tpr@1fpr", "TPR @ 1% FPR")]:
    mm_mat = gen_to_matrix(gen_mm, metric, DATASET_NAMES)
    lr_mat = gen_to_matrix(gen_lr, metric, DATASET_NAMES)
    dct_mat = gen_to_matrix(gen_dct, metric, DATASET_NAMES)

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=["MM Probe", "LR Probe", f"Best DCT"],
        horizontal_spacing=0.08,
    )

    for col, mat in enumerate([mm_mat, lr_mat, dct_mat], start=1):
        fig.add_trace(go.Heatmap(
            z=mat,
            x=labels_display,
            y=labels_display,
            text=[[f"{v:.3f}" for v in row] for row in mat],
            texttemplate="%{text}",
            colorscale="RdYlGn",
            zmin=0, zmax=1,
            showscale=(col == 3),
            colorbar=dict(title=metric_title, thickness=15),
        ), row=1, col=col)

    fig.update_layout(
        **PLOT_BASE,
        title=(
            f"Experiment 2: Generalization Matrix — {metric_title}\n"
            "<sup>Rows = Train dataset &nbsp; Cols = Test dataset</sup>"
        ),
        height=400, width=1060,
    )
    fig.update_yaxes(title_text="Train dataset", tickangle=45)
    for col in range(1, 4):
        fig.update_xaxes(title_text="Test dataset", row=1, col=col)
    fig.show()
    metric_slug = metric.replace("@", "at").replace("%", "pct").replace(" ", "_")
    save_fig(fig, f"exp2_generalization_{metric_slug}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Saved: dct_probes/images/llama-3.1-8b-32samples-fewshot-lying-categorical/exp2_generalization_auroc.png


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Saved: dct_probes/images/llama-3.1-8b-32samples-fewshot-lying-categorical/exp2_generalization_tprat1fpr.png


---
## Experiment 3: Ensemble Probe k-Sweep

Sweep over k (number of top DCT directions) for mean and max ensemble probes calibrated on Cities.
Compare AUROC and TPR@1%FPR against MM and LR baselines.

In [48]:
K_VALUES = [k for k in [1, 2, 3, 5, 8, 10, 15, 20, 30, 50] if k <= V.shape[1]]

# Directions ranked by judge score
# all_indices_by_judge is already defined in cell-train-baselines

sweep: dict[str, list] = {
    "k": K_VALUES,
    "mean_train_auroc": [], "mean_test_auroc": [],
    "mean_train_tpr1": [], "mean_test_tpr1": [],
    "max_train_auroc": [], "max_test_auroc": [],
    "max_train_tpr1": [], "max_test_tpr1": [],
}

for k in tqdm(K_VALUES, desc="k-sweep"):
    top_idx = all_indices_by_judge[:k]
    directions = V[:, top_idx].float()

    for agg in ["mean", "max"]:
        probe = FixedEnsembleProbe(directions, aggregation=agg)
        train_probe(probe, train_acts["cities"], train_labels["cities"])

        tr = eval_probe(probe, train_acts["cities"], train_labels["cities"])
        te = eval_probe(probe, test_acts["cities"], test_labels["cities"])

        sweep[f"{agg}_train_auroc"].append(tr["auroc"])
        sweep[f"{agg}_test_auroc"].append(te["auroc"])
        sweep[f"{agg}_train_tpr1"].append(tr["tpr@1fpr"])
        sweep[f"{agg}_test_tpr1"].append(te["tpr@1fpr"])

# Summary table
sweep_df = pd.DataFrame({
    "k": sweep["k"],
    "Mean Ens Train AUROC": [f"{v:.3f}" for v in sweep["mean_train_auroc"]],
    "Mean Ens Test AUROC": [f"{v:.3f}" for v in sweep["mean_test_auroc"]],
    "Max Ens Train AUROC": [f"{v:.3f}" for v in sweep["max_train_auroc"]],
    "Max Ens Test AUROC": [f"{v:.3f}" for v in sweep["max_test_auroc"]],
})
display(sweep_df)

k-sweep: 100%|██████████| 10/10 [00:06<00:00,  1.51it/s]


,k,Mean Ens Train AUROC,Mean Ens Test AUROC,Max Ens Train AUROC,Max Ens Test AUROC
0,1,0.535,0.557,0.535,0.557
1,2,0.619,0.628,0.535,0.557
2,3,0.836,0.812,0.535,0.557
3,5,0.818,0.792,0.535,0.557
4,8,0.986,0.977,0.535,0.557
5,10,0.990,0.981,0.535,0.557
6,15,0.994,0.990,0.535,0.557
7,20,0.994,0.990,0.793,0.778
8,30,0.995,0.992,0.793,0.778
9,50,0.993,0.991,0.793,0.778


In [49]:
mm_test_cities = eval_probe(mm_probe, test_acts["cities"], test_labels["cities"])
lr_test_cities = eval_probe(lr_probe, test_acts["cities"], test_labels["cities"])

for metric_key, metric_title in [("auroc", "AUROC"), ("tpr1", "TPR @ 1% FPR")]:
    baseline_metric = "auroc" if metric_key == "auroc" else "tpr@1fpr"
    mm_val = mm_test_cities[baseline_metric]
    lr_val = lr_test_cities[baseline_metric]

    fig = go.Figure()

    for agg, color, label in [
        ("mean", PLOT_COLORS["ensemble_mean"], "Mean Ensemble"),
        ("max", PLOT_COLORS["ensemble_max"], "Max Ensemble"),
    ]:
        fig.add_trace(go.Scatter(
            x=sweep["k"], y=sweep[f"{agg}_train_{metric_key}"],
            mode="lines+markers", name=f"{label} (Train)",
            line=dict(color=color, width=1.8, dash="dot"),
            marker=dict(size=6, symbol="square"),
        ))
        fig.add_trace(go.Scatter(
            x=sweep["k"], y=sweep[f"{agg}_test_{metric_key}"],
            mode="lines+markers", name=f"{label} (Test)",
            line=dict(color=color, width=2.5),
            marker=dict(size=7),
        ))

    fig.add_hline(
        y=mm_val, line_dash="dash", line_color=PLOT_COLORS["mm"],
        annotation_text=f"MM ({mm_val:.3f})",
        annotation_position="bottom right", annotation_font_size=12,
    )
    fig.add_hline(
        y=lr_val, line_dash="dash", line_color=PLOT_COLORS["lr"],
        annotation_text=f"LR ({lr_val:.3f})",
        annotation_position="top right", annotation_font_size=12,
    )

    fig.update_layout(
        **PLOT_BASE,
        title=f"Experiment 3: Ensemble {metric_title} vs k (Test Set — Cities)",
        xaxis_title="k (number of DCT directions)",
        yaxis_title=metric_title,
        yaxis_range=[0, 1.12],
        height=460, width=820,
    )
    fig.show()
    save_fig(fig, f"exp3_ensemble_sweep_{metric_key}")

best_k_mean = K_VALUES[int(np.argmax(sweep["mean_test_auroc"]))]
best_k_max = K_VALUES[int(np.argmax(sweep["max_test_auroc"]))]
print(f"\nBest k (Mean Ensemble) by test AUROC: k={best_k_mean}  ({max(sweep['mean_test_auroc']):.3f})")
print(f"Best k (Max Ensemble)  by test AUROC: k={best_k_max}  ({max(sweep['max_test_auroc']):.3f})")
print(f"MM baseline test AUROC: {mm_val:.3f}")
print(f"LR baseline test AUROC: {lr_val:.3f}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Saved: dct_probes/images/llama-3.1-8b-32samples-fewshot-lying-categorical/exp3_ensemble_sweep_auroc.png


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Saved: dct_probes/images/llama-3.1-8b-32samples-fewshot-lying-categorical/exp3_ensemble_sweep_tpr1.png

Best k (Mean Ensemble) by test AUROC: k=30  (0.992)
Best k (Max Ensemble)  by test AUROC: k=20  (0.778)
MM baseline test AUROC: 0.898
LR baseline test AUROC: 0.986


---
## Experiment 4: Ensemble Generalization

Using the best k found above, evaluate how mean and max ensemble probes generalise across all three datasets.

In [50]:
# Use best k by test AUROC (pick mean ensemble as primary, max as secondary)
best_k = best_k_mean
top_idx_best = all_indices_by_judge[:best_k]
directions_best = V[:, top_idx_best].float()
print(f"Using k={best_k}, factor indices (by judge score): {top_idx_best}")

# Verify performance on cities test
for agg in ["mean", "max"]:
    probe = FixedEnsembleProbe(directions_best, aggregation=agg)
    train_probe(probe, train_acts["cities"], train_labels["cities"])
    r = eval_probe(probe, test_acts["cities"], test_labels["cities"])
    print(f"  {agg.capitalize()} ensemble (k={best_k}): AUROC={r['auroc']:.3f}, TPR@1%FPR={r['tpr@1fpr']:.3f}")

Using k=30, factor indices (by judge score): [435, 62, 72, 218, 24, 375, 431, 198, 249, 484, 178, 324, 333, 141, 154, 345, 113, 428, 101, 504, 408, 241, 398, 264, 194, 415, 315, 115, 399, 149]
  Mean ensemble (k=30): AUROC=0.992, TPR@1%FPR=0.871
  Max ensemble (k=30): AUROC=0.778, TPR@1%FPR=0.054


In [51]:
def make_ens_probe(agg: str):
    def factory(acts, labels):
        probe = FixedEnsembleProbe(directions_best, aggregation=agg)
        train_probe(probe, acts, labels)
        return probe
    return factory

gen_ens_mean = build_generalization_data(
    make_ens_probe("mean"),
    train_acts, train_labels, test_acts, test_labels, DATASET_NAMES,
)
gen_ens_max = build_generalization_data(
    make_ens_probe("max"),
    train_acts, train_labels, test_acts, test_labels, DATASET_NAMES,
)
print("Ensemble generalization computed.")

Ensemble generalization computed.


In [52]:
for metric, metric_title in [("auroc", "AUROC"), ("tpr@1fpr", "TPR @ 1% FPR")]:
    mm_mat = gen_to_matrix(gen_mm, metric, DATASET_NAMES)
    ens_mean_mat = gen_to_matrix(gen_ens_mean, metric, DATASET_NAMES)
    ens_max_mat = gen_to_matrix(gen_ens_max, metric, DATASET_NAMES)

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=["MM Probe", f"Mean Ensemble (k={best_k})", f"Max Ensemble (k={best_k})"],
        horizontal_spacing=0.08,
    )

    for col, mat in enumerate([mm_mat, ens_mean_mat, ens_max_mat], start=1):
        fig.add_trace(go.Heatmap(
            z=mat,
            x=labels_display,
            y=labels_display,
            text=[[f"{v:.3f}" for v in row] for row in mat],
            texttemplate="%{text}",
            colorscale="RdYlGn",
            zmin=0, zmax=1,
            showscale=(col == 3),
            colorbar=dict(title=metric_title, thickness=15),
        ), row=1, col=col)

    fig.update_layout(
        **PLOT_BASE,
        title=(
            f"Experiment 4: Ensemble Generalization — {metric_title}\n"
            "<sup>Rows = Train dataset &nbsp; Cols = Test dataset</sup>"
        ),
        height=400, width=1060,
    )
    fig.update_yaxes(title_text="Train dataset", tickangle=45)
    for col in range(1, 4):
        fig.update_xaxes(title_text="Test dataset", row=1, col=col)
    fig.show()
    metric_slug = metric.replace("@", "at").replace("%", "pct").replace(" ", "_")
    save_fig(fig, f"exp4_ensemble_generalization_{metric_slug}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Saved: dct_probes/images/llama-3.1-8b-32samples-fewshot-lying-categorical/exp4_ensemble_generalization_auroc.png


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Saved: dct_probes/images/llama-3.1-8b-32samples-fewshot-lying-categorical/exp4_ensemble_generalization_tprat1fpr.png


---
## Experiment 5: DCT Subspace vs Random Subspace

Do random directions work as well as DCT directions when given the correct magnitude and bias?

For each k, we project activations into a k-dimensional subspace and train LR. We compare:
- **DCT subspace**: top-k directions ranked by train AUROC
- **Random subspaces**: many random k-d subspaces (faint lines) + their mean (solid line)

In [53]:
N_RANDOM_TRIALS = 20
K_VALUES_EXP5 = [k for k in [1, 2, 3, 5, 8, 10, 15, 20, 30, 50] if k <= V.shape[1]]

train_a = train_acts["cities"]
train_l = train_labels["cities"]
test_a  = test_acts["cities"]
test_l  = test_labels["cities"]
d_model = train_a.shape[1]

def subspace_lr_auroc(train_proj, test_proj, train_l, test_l):
    scaler = StandardScaler()
    train_proj = scaler.fit_transform(train_proj)
    test_proj  = scaler.transform(test_proj)
    lr = LogisticRegression(max_iter=1000)
    lr.fit(train_proj, train_l.numpy())
    scores = lr.predict_proba(test_proj)[:, 1]
    return roc_auc_score(test_l.numpy(), scores)

# DCT subspace sweep — directions ranked by judge score
dct_aurocs = []
for k in K_VALUES_EXP5:
    top_idx = all_indices_by_judge[:k]
    Q, _ = torch.linalg.qr(V[:, top_idx].float())
    train_proj = (train_a.float() @ Q).detach().numpy()
    test_proj  = (test_a.float()  @ Q).detach().numpy()
    dct_aurocs.append(subspace_lr_auroc(train_proj, test_proj, train_l, test_l))

# Random subspace sweep — one list of AUROCs per trial
random_aurocs_by_trial = []
for _ in tqdm(range(N_RANDOM_TRIALS), desc="random trials"):
    trial = []
    for k in K_VALUES_EXP5:
        Q, _ = torch.linalg.qr(torch.randn(d_model, k))
        train_proj = (train_a.float() @ Q).detach().numpy()
        test_proj  = (test_a.float()  @ Q).detach().numpy()
        trial.append(subspace_lr_auroc(train_proj, test_proj, train_l, test_l))
    random_aurocs_by_trial.append(trial)

random_mean = np.mean(random_aurocs_by_trial, axis=0)

# Full-space LR baseline
scaler_full = StandardScaler()
lr_full = LogisticRegression(max_iter=1000)
lr_full.fit(scaler_full.fit_transform(train_a.numpy()), train_l.numpy())
full_scores = lr_full.predict_proba(scaler_full.transform(test_a.numpy()))[:, 1]
full_auroc = roc_auc_score(test_l.numpy(), full_scores)

# --- Plot ---
fig = go.Figure()

# Faint random trial lines
for i, trial in enumerate(random_aurocs_by_trial):
    fig.add_trace(go.Scatter(
        x=K_VALUES_EXP5, y=trial,
        mode="lines",
        line=dict(color=PLOT_COLORS["random"], width=1),
        opacity=0.25,
        showlegend=(i == 0),
        name="Random (individual trials)",
    ))

# Mean random line
fig.add_trace(go.Scatter(
    x=K_VALUES_EXP5, y=random_mean,
    mode="lines+markers",
    name="Random (mean)",
    line=dict(color=PLOT_COLORS["random"], width=2.5, dash="dot"),
    marker=dict(size=7),
))

# DCT line
fig.add_trace(go.Scatter(
    x=K_VALUES_EXP5, y=dct_aurocs,
    mode="lines+markers",
    name="DCT subspace (judge-ranked)",
    line=dict(color=PLOT_COLORS["dct"], width=2.5),
    marker=dict(size=7),
))

fig.add_hline(
    y=full_auroc, line_dash="dash", line_color=PLOT_COLORS["lr"],
    annotation_text=f"Full LR AUROC ({full_auroc:.3f})", annotation_font_size=13,
)

fig.update_layout(
    **PLOT_BASE,
    title="LR AUROC: DCT Subspace (judge-ranked) vs Random Subspace vs Full Space",
    xaxis_title="k (subspace dimensions)",
    yaxis_title="Test AUROC",
    height=500,
    width=900,
)
fig.show()
save_fig(fig, "exp5_dct_vs_random_subspace")

random trials: 100%|██████████| 20/20 [00:00<00:00, 34.48it/s]


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Saved: dct_probes/images/llama-3.1-8b-32samples-fewshot-lying-categorical/exp5_dct_vs_random_subspace.png


---
## Experiment 6: Does Judge Score Rank Align with Probe Quality?

If the judge score ranking is meaningful, directions ranked higher by the judge should also have higher AUROC and TPR@1%FPR as probes.

We test this with **Spearman rank correlation** between judge score and probe metrics across all DCT factors. A statistically significant positive correlation is evidence the method is working; a near-zero or insignificant correlation suggests judge scores add no signal beyond random selection.

In [54]:
from scipy import stats

# --- Compute AUROC and TPR@1%FPR for every DCT factor on Cities test set ---
all_factor_metrics: list[dict] = []
for idx in tqdm(range(V.shape[1]), desc="per-factor eval"):
    probe = DirectionalProbe(V[:, idx].float())
    train_probe(probe, train_acts["cities"], train_labels["cities"])
    r = eval_probe(probe, test_acts["cities"], test_labels["cities"])
    judge_score = float(mean_deltas[idx]) if idx in mean_deltas.index else float("nan")
    all_factor_metrics.append({
        "factor_idx": idx,
        "judge_score": judge_score,
        "auroc": r["auroc"],
        "tpr@1fpr": r["tpr@1fpr"],
    })

df_factors = pd.DataFrame(all_factor_metrics).dropna(subset=["judge_score"])

# --- Spearman rank correlation ---
rho_auroc, p_auroc = stats.spearmanr(df_factors["judge_score"], df_factors["auroc"])
rho_tpr,   p_tpr   = stats.spearmanr(df_factors["judge_score"], df_factors["tpr@1fpr"])

print("Spearman rank correlation: judge score vs probe metric")
print(f"  vs AUROC:      ρ = {rho_auroc:+.3f}  (p = {p_auroc:.4f}{'  ***' if p_auroc < 0.001 else '  **' if p_auroc < 0.01 else '  *' if p_auroc < 0.05 else '  n.s.'})")
print(f"  vs TPR@1%FPR:  ρ = {rho_tpr:+.3f}  (p = {p_tpr:.4f}{'  ***' if p_tpr < 0.001 else '  **' if p_tpr < 0.01 else '  *' if p_tpr < 0.05 else '  n.s.'})")
print()
print("Interpretation:")
if p_auroc < 0.05 and rho_auroc > 0:
    print("  ✓ Significant positive correlation with AUROC — judge ranking is informative.")
elif p_auroc < 0.05 and rho_auroc < 0:
    print("  ✗ Significant *negative* correlation with AUROC — judge ranking is anti-correlated!")
else:
    print("  ✗ No significant correlation with AUROC — judge ranking adds no probe-quality signal.")

if p_tpr < 0.05 and rho_tpr > 0:
    print("  ✓ Significant positive correlation with TPR@1%FPR — judge ranking is informative.")
elif p_tpr < 0.05 and rho_tpr < 0:
    print("  ✗ Significant *negative* correlation with TPR@1%FPR — judge ranking is anti-correlated!")
else:
    print("  ✗ No significant correlation with TPR@1%FPR — judge ranking adds no probe-quality signal.")

per-factor eval: 100%|██████████| 512/512 [01:28<00:00,  5.80it/s]

Spearman rank correlation: judge score vs probe metric
  vs AUROC:      ρ = +0.080  (p = 0.0688  n.s.)
  vs TPR@1%FPR:  ρ = +0.094  (p = 0.0341  *)

Interpretation:
  ✗ No significant correlation with AUROC — judge ranking adds no probe-quality signal.
  ✓ Significant positive correlation with TPR@1%FPR — judge ranking is informative.


In [55]:
# --- Scatter plots: judge score vs AUROC and TPR@1%FPR ---
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f"Judge Score vs AUROC  (ρ={rho_auroc:+.3f}, p={p_auroc:.4f})",
        f"Judge Score vs TPR@1%FPR  (ρ={rho_tpr:+.3f}, p={p_tpr:.4f})",
    ],
    horizontal_spacing=0.12,
)

for col, metric in enumerate(["auroc", "tpr@1fpr"], start=1):
    x = df_factors["judge_score"]
    y = df_factors[metric]

    # OLS trend line
    slope, intercept, *_ = stats.linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), 100)

    fig.add_trace(go.Scatter(
        x=x, y=y,
        mode="markers",
        marker=dict(
            color=df_factors["factor_idx"],
            colorscale="Viridis",
            size=7,
            opacity=0.75,
            showscale=(col == 2),
            colorbar=dict(title="Factor idx", thickness=12),
        ),
        # text=[f"Factor #{i}" for i in df_factors["factor_idx"]],
        hovertemplate="<b>%{text}</b><br>Judge: %{x:.3f}<br>" + metric + ": %{y:.3f}<extra></extra>",
        showlegend=False,
    ), row=1, col=col)

    fig.add_trace(go.Scatter(
        x=x_line, y=slope * x_line + intercept,
        mode="lines",
        line=dict(color="#EF4444", width=2, dash="dash"),
        name="OLS trend",
        showlegend=(col == 1),
    ), row=1, col=col)

fig.update_layout(
    **PLOT_BASE,
    title="Experiment 6: Judge Score vs Probe Quality (all DCT factors, Cities test set)",
    height=460, width=1060,
    xaxis_title="Judge score", xaxis2_title="Judge score",
    yaxis_title="AUROC", yaxis2_title="TPR @ 1% FPR",
    legend_x=0, legend_y=1,  # move legend to top-left, away from colorbar
)

fig.show()
save_fig(fig, "exp6_judge_score_vs_probe_quality")

# --- Summary table: top-10 and bottom-10 by judge score ---
df_sorted = df_factors.sort_values("judge_score", ascending=False)
print("Top 10 by judge score:")
display(df_sorted.head(10).round(3).to_string(index=False))
print("\nBottom 10 by judge score:")
display(df_sorted.tail(10).round(3).to_string(index=False))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Saved: dct_probes/images/llama-3.1-8b-32samples-fewshot-lying-categorical/exp6_judge_score_vs_probe_quality.png
Top 10 by judge score:


' factor_idx  judge_score  auroc  tpr@1fpr\n        435          0.4  0.557     0.020\n         62          0.3  0.611     0.027\n         72          0.3  0.905     0.259\n        218          0.3  0.428     0.014\n         24          0.3  0.429     0.007\n        375          0.3  0.896     0.102\n        431          0.3  0.933     0.469\n        198          0.3  0.960     0.061\n        249          0.3  0.641     0.048\n        484          0.3  0.723     0.068'


Bottom 10 by judge score:


' factor_idx  judge_score  auroc  tpr@1fpr\n        252          0.0  0.653     0.027\n        297          0.0  0.736     0.061\n        295          0.0  0.525     0.027\n        108          0.0  0.554     0.000\n        105          0.0  0.525     0.000\n        292          0.0  0.562     0.020\n        391          0.0  0.618     0.048\n        392          0.0  0.839     0.177\n        207          0.0  0.429     0.000\n        511          0.0  0.873     0.265'